In [6]:
import pandas as pd
import numpy as np

In [7]:
df=pd.read_csv("DateFruit_Dataset.csv")

In [8]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [9]:
df.shape

(898, 35)

In [10]:
X = df.drop("Class",axis=1)
y=df["Class"]

In [11]:
X.head()


,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,SkewRB,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,0.6019,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,0.4134,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,0.9183,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,1.8028,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,0.8865,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666


In [12]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le =LabelEncoder()
y=le.fit_transform(y)

In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [14]:
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [16]:
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)


In [17]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)


In [18]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [21]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model=nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    def forward(self,x):
        return self.model(x)
    
    

In [22]:
model=ANN()

criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [24]:
epochs=100
for epoch in range(epochs):
    model.train()

    running_loss=0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()

        outputs=model(xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()

    train_loss=running_loss/len(train_loader)
    print(f"epoch={epoch+1}/{epoch},loss={train_loss}")

epoch=1/0,loss=0.03182018918754614
epoch=2/1,loss=0.030855476147616686
epoch=3/2,loss=0.028471985001764868
epoch=4/3,loss=0.030687457335222025
epoch=5/4,loss=0.030670141720253487
epoch=6/5,loss=0.029102486720227676
epoch=7/6,loss=0.038133219172975616
epoch=8/7,loss=0.030263382961730593
epoch=9/8,loss=0.02705802714816578
epoch=10/9,loss=0.028343449868059353
epoch=11/10,loss=0.033148991875350475
epoch=12/11,loss=0.0329759910951986
epoch=13/12,loss=0.023860445721884786
epoch=14/13,loss=0.026106772104116237
epoch=15/14,loss=0.02497501620698882
epoch=16/15,loss=0.0202264783843218
epoch=17/16,loss=0.01997864917528046
epoch=18/17,loss=0.021437720436116924
epoch=19/18,loss=0.017928935719006087
epoch=20/19,loss=0.020473423540470718
epoch=21/20,loss=0.021351108206031116
epoch=22/21,loss=0.019469988998025656
epoch=23/22,loss=0.019699113241032414
epoch=24/23,loss=0.01861498101978846
epoch=25/24,loss=0.018005286809056997
epoch=26/25,loss=0.01608028184905972
epoch=27/26,loss=0.015474445054955457
epo

In [30]:
model.eval()

total=0
correct=0
with torch.no_grad():
    for xbyb in test_loader:
        outputs=model(xb)
        _, predicted=torch.max(outputs,1)

        correct+=(predicted == yb).sum().item()
        total+=yb.size(0)
print("accuracy: ",correct/total *100)

accuracy:  100.0


In [28]:
y_test.shape

(180,)